# Ke Jin — alert policy and automated checks

Frozen model: LightGBM no_dc, run aki_ml_v1_28e0b1fb83, Platt calibration. This notebook runs the artifact/leakage checks and a prespecified alert-policy experiment without retraining.

## Current version: v3 (`configs/alert_policy_v3.yaml`)

- **Risks.** Selection and confirmation use validation risks recomputed with the batch-independent feature engine `patient_window_v2` and the unchanged weights (`scripts/rescore_validation_feature_v2.py`, report in `artifacts/handoff/feature_v2/`). The legacy engine gave a patient different features depending on which other patients were in the same batch; v2 gives identical features for a whole batch, a single patient and events cut at each origin, and validation AUROC is unchanged to 4 decimals.
- **Roles.** Deterministic patient-level validation split: 70% selection, 30% confirmation (same roles as v2). Train OOF risks exist only for the legacy engine and are not used.
- **Candidates.** The 12 v2 rules plus multi-horizon confirmation (12 h risk >= 0.5 x threshold) and a stricter early threshold (+0.05 at ICU hours 1-6), each with 6 h and 12 h cooldown: 16 rules x 31 thresholds.
- **Caveat.** The full validation set was viewed in Demo I, and the confirmation patients were used once in v2. Confirmation is an internal holdout, **not an untouched final test or clinical validation**. No test features/labels are loaded.

v2 (`configs/alert_policy_v1.yaml`, legacy-engine risks) is superseded; its results are reported in `handoff/KEJIN-EVALUATION.md` and were not reproduced here.

## Modules and safeguards

- Checks A1-A12: src/aki_eval/checks.py.
- Fixed-origin cutoff replay: src/aki_eval/replay.py (v3: patient_window_v2 with float64 admission inputs).
- Stateful alert events, separate from risk probabilities: src/aki_eval/policies.py.
- Immutable local prespec and choice records: src/aki_eval/protocol.py.
- Orchestration: src/aki_eval/study.py; this notebook is a thin, rerunnable entry point.

Aggregate outputs go to `artifacts/handoff/policy_v3/<run>/`; `LATEST.json` names the run.

In [ ]:
from pathlib import Path
import sys
from threadpoolctl import threadpool_limits

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "alert_policy_v1.yaml").is_file())
sys.path.insert(0, str(REPO_ROOT / "src"))
from aki_eval.study import KeJinStudy

THREAD_LIMIT = threadpool_limits(limits=2)
study = KeJinStudy(REPO_ROOT, config_path=REPO_ROOT / "configs" / "alert_policy_v3.yaml")
print("Run:", study.out.name)
print("Prespec SHA256:", study.data.prespec["prespec_sha256"])


## Part 0 — Reproduce frozen Demo I

Both train-OOF and full-validation alert metrics must match the saved policy before the new experiment proceeds. This baseline reproduction does not tune any candidate.


In [ ]:
display(study.reproduce_baseline())


## Part A — Artifact and temporal checks

A1 unique keys; A2 probabilities; A3 schema/version/model/config hashes; A4 prediction coverage; A5 patient splits; A6 selected stay mapping; A7 chronology; A8 label windows/eligibility; A9 pre-onset origins; A10 at least 2,000 feature replays plus future injection; A11 urine-specific future truncation; A12 fitting declarations and actual OOF fold roles.

Failure stops policy selection. Details contain counts only. A10 independently replays dynamic features but receives admission-static/service inputs from recorded snapshots; raw source timing for those inputs is unavailable. A12 checks historical manifest assertions and saved fold roles; it does not independently re-audit inaccessible training data.


In [ ]:
display(study.checks())


## Part B.1 — Immutable prespecification

The config, data/model hashes and evaluation-code hashes are written before a candidate grid is evaluated. Existing records cannot be replaced with different inputs, settings or code. The finite grid compares sustain, cooldown, episode suppression, substantial-risk-rise repeat, multi-horizon confirmation (12 h risk >= 0.5 x threshold, the uniform-hazard share of the 24 h risk) and stricter-early-threshold (+0.05 at ICU hours 1-6) rules.

Hourly k-score sustain means observed high duration of at least k-1 h. Gaps reset sustain and episodes. Early missing input is unscorable, not zero risk.

Selection minimises false alerts per scorable monitored patient-day under detection, lead, PPV, alert-burden and cutoff-stability constraints. Non-target-event patient burden cannot increase. Ties favour fewer affected non-target-event patients, then simpler rules and higher thresholds.


In [ ]:
display(study.prespecification())


## Part B.2 — Validation selection and training-OOF diagnostics

Every choice is made using validation-selection patients. A zero-lag replay control and 1/2 h input cutoffs keep prediction origin and label window fixed. At zero lag the replay must reproduce the evaluated risks exactly. Rolling windows stay anchored at origin; laboratory reporting delay still applies. Hourly service uses its last recorded cutoff-visible state; raw service-event timing is not supplied.

Cutoff comparisons report paired coverage, probability changes, threshold flips, alert-event flips and patient alert-presence flips. Both point-warning and patient-alert stability affect selection. The risk model and calibrated probabilities are never trained or altered.

v3 has no train-OOF diagnostic (OOF risks exist only for the legacy engine). Validation-selection grids, per-rule representatives and the trade-off figure are saved. No feasible candidate is reported honestly rather than relaxing criteria.


In [ ]:
display(study.select())


## Part B.3 — Fixed confirmation and paired patient bootstrap

Compare only the sealed candidate and frozen Demo I on the other validation patients. Use 1,000 **paired patient-level** bootstrap resamples, including confidence intervals for metric differences. Confirmation checks the same prespecified constraints and cutoff sensitivity. Failure does not trigger retuning.

A completed confirmation is reused on a rerun with the same immutable choice. Different settings require a new labelled development cycle and appropriate untouched evaluation data; rerunning code does not create a new independent validation sample.

The candidate JSON always has threshold.locked=false and requires a stateful alert adapter. The current live loader rejects it, preventing a sustained/cooldown/episode policy from silently degrading into the website's instantaneous threshold comparison. Original policy.json, fitted weights and src/aki_ml remain unchanged.


In [ ]:
display(study.confirm())


## Summary and hand-back

The summary reports actual trade-offs and the confirmation outcome. Definitions match original Demo I: exposure is scorable monitored hours / 24, unknown-outcome alerts count toward burden but not PPV, and non_aki denotes patients without a target event in the evaluated monitoring/horizon risk set.

Source notebook outputs are cleared for sharing. Executed aggregate-only copies and versioned CSV/JSON/PNG files stay local under artifacts/handoff/kejin/. The root LATEST.json identifies the run. Required files include checks.csv, prespec.json, choice.json, both OOF diagnostic tables, validation-selection tables, policy_confirmation_validation.csv, policy_v1_candidate.json, cutoff tables, trade-off figures and SUMMARY.md.

No final-test evaluation or live clinical-policy activation is claimed.


In [ ]:
print(study.summary())
from aki_eval.reporting import finalize_reports
print(finalize_reports(study.out))
from IPython.display import Image, display
display(Image(filename=str(study.out / "policy_tradeoff_validation_selection.png")))


## Recorded v3 result — validation-policy-v3-66054ed84c02e348

- Part 0 reproduced Demo I on the stored risks; A1-A12 passed. Zero-lag replay reproduced all 31,694 selection and 15,999 confirmation risks exactly.
- Validation roles: 3,420 selection patients, 1,530 confirmation patients.
- Selected: `confirm12_cool12` — 24 h risk >= 0.21 and 12 h risk >= 0.105, 12 h cooldown (model unchanged).
- Confirmation versus demo-v1 on the same risks (1,000 paired patient bootstraps):

| Metric | demo-v1 | Candidate | Difference (95% CI) |
|---|---:|---:|---:|
| False alerts / patient-day | 0.424 | 0.230 | -0.194 (-0.219 to -0.171) |
| Alerts / patient-day | 0.804 | 0.423 | -0.381 |
| Alert PPV | 33.7% | 33.9% | +0.2 pp (-0.9 to +1.3) |
| AKI event detection | 76.9% | 71.9% | -5.1 pp (-7.7 to -2.7) |
| Median lead time | 12.8 h | 11.7 h | -1.1 h |
| Non-AKI patients alerted | 35.8% | 31.1% | -4.7 pp |

- Outcome: **not confirmed**. The prespecified maximum detection loss is 5 pp; the observed loss was 5.08 pp (limit 71.95%, observed 71.86% of 295 events). All other criteria passed. No retuning; the candidate stays unlocked and the website keeps demo-v1.
- Final test and clinical evaluation were not performed.